### Large Language Models (LLMs)

In [2]:
import torch
import torch.nn as nn
import numpy as np
import tiktoken
import torch.nn.functional as F

### 1. Tokenization

In [3]:
context_size = 1024
vocab_size = 50257
token_emb = 768
pos_emb = 768
n_layers = 12
drop_rate = .2
qkv_bias = False

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

In [5]:
text = "How are you doing today?"
tokenizer.encode(text)

[2437, 389, 345, 1804, 1909, 30]

In [6]:
class SelfAttentionMechanism(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
        self.W_key = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
        self.W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

    def forward(self, x: torch.Tensor):
        query = x @ self.W_query
        key = x @ self.W_key
        value = x @ self.W_value

        weighted_vec = query @ key.T

        scaled_weighted_vec = weighted_vec / key.shape[-1]**.5

        mask = torch.triu(torch.ones_like(scaled_weighted_vec), diagonal=1)

        masked_vec = scaled_weighted_vec.masked_fill(mask.bool(), -torch.inf)

        softmax = F.softmax(masked_vec, dim=-1)

        attn_mat = softmax @ value

        return attn_mat





In [7]:
device = "gpu" if torch.cuda.is_available() else "cpu"
device

'gpu'

In [8]:
mna = SelfAttentionMechanism(155000,500)
mna.forward(torch.rand(size=(1500,155000)))

tensor([[38923.5625, 38761.7109, 38760.1055,  ..., 38919.4805, 38869.5156,
         38750.2812],
        [38931.3711, 38748.5156, 38796.1875,  ..., 38901.8750, 38891.5234,
         38709.2578],
        [38931.3711, 38748.5156, 38796.1875,  ..., 38901.8750, 38891.5234,
         38709.2578],
        ...,
        [39025.4375, 38913.7969, 38928.9336,  ..., 39020.4219, 39075.6289,
         38920.8438],
        [39025.4375, 38913.7969, 38928.9336,  ..., 39020.4219, 39075.6289,
         38920.8438],
        [39025.4375, 38913.7969, 38928.9336,  ..., 39020.4219, 39075.6289,
         38920.8438]])

In [9]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_layers, d_in, d_out):
        super().__init__()
        self.layers = nn.ModuleList(
            [SelfAttentionMechanism(d_in, d_out) for _ in range(n_layers)]
        )

    def forward(self, x):
        return torch.cat([layer(x) for layer in self.layers], dim=-1)

In [12]:
asa = MultiHeadAttention(10, 500000, 50)
asa.forward(torch.rand(size=(1011, 500000)))

tensor([[125037.3984, 125117.2188, 125110.3125,  ..., 125041.1406,
         125042.9062, 125126.6953],
        [125099.7734, 125102.8281, 125132.1172,  ..., 125179.8438,
         125225.8594, 125299.5234],
        [125099.7734, 125102.8281, 125132.1172,  ..., 125179.8438,
         125225.8594, 125299.5234],
        ...,
        [125387.3125, 125323.8203, 125370.0000,  ..., 125346.8047,
         125391.5859, 125347.4922],
        [125387.3125, 125323.8203, 125370.0000,  ..., 125346.8047,
         125391.5859, 125347.4922],
        [125387.3125, 125323.8203, 125370.0000,  ..., 125346.8047,
         125391.5859, 125347.4922]])

<link href="https://github.com" style="color:blue;" >github</link>